# Kartoffelmachine — Segunda prueba de clasificación (TRL 3)

Prueba exploratoria con cuatro imágenes de distintos colores y fondos; dos con brotación visible. Las etiquetas son visuales y no reemplazan una validación con muestras independientes.


In [ ]:
%pip -q install ultralytics ncnn pandas pillow matplotlib


In [ ]:
from pathlib import Path
from google.colab import files
from ultralytics import YOLO
import pandas as pd, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from zipfile import ZipFile

# El modelo ya se extrajo al ejecutar el cuaderno anterior.
model_dir = Path('/content/trl3_ia/best_ncnn_model')
if not (model_dir/'model.ncnn.param').exists():
    raise FileNotFoundError('Primero ejecuta las dos primeras celdas del cuaderno anterior para cargar TRL3_Paso11_Datos_IA.zip.')

print('Sube TRL3_Imagenes_Nuevas_IA.zip:')
uploaded = files.upload()
zip_file = next((Path(n) for n in uploaded if n.endswith('.zip')), None)
if zip_file is None: raise ValueError('No se recibió un archivo ZIP.')
photos_dir = Path('/content/trl3_ia/imagenes_nuevas')
photos_dir.mkdir(parents=True,exist_ok=True)
with ZipFile(zip_file) as z: z.extractall(photos_dir)

model = YOLO(str(model_dir), task='classify')
expected = {
 'IA2_BE_01.png':'Buen estado',
 'IA2_BE_02.png':'Buen estado',
 'IA2_D_Brotes_01.png':'Defectuoso',
 'IA2_D_Brotes_02.png':'Defectuoso'
}
rows=[]
fig, axes=plt.subplots(1,4,figsize=(15,4.2))
for ax,(filename,reference) in zip(axes,expected.items()):
    path=photos_dir/filename
    if not path.exists():raise FileNotFoundError(path)
    pred=model.predict(source=str(path),imgsz=224,device='cpu',verbose=False)[0]
    probs=pred.probs.data.cpu().numpy().astype(float)
    idx=int(np.argmax(probs))
    label=pred.names[idx]
    # Normalización ligera para comparar las etiquetas, no transforma predicciones.
    canon=lambda s: ''.join(c for c in s.lower().strip() if c.isalnum())
    matches=canon(label)==canon(reference)
    rows.append({'imagen':filename,'referencia_visual':reference,'prediccion_modelo':label,
                 'confianza':round(float(probs[idx]),6),'coincide':matches})
    ax.imshow(Image.open(path).convert('RGB'))
    ax.set_title(f'{filename}\nRef: {reference}\nPred: {label}\nConfianza: {probs[idx]:.2%}',fontsize=9)
    ax.axis('off')
plt.tight_layout()
image_out='/content/TRL3_Prueba2_Comparacion_IA.png'
fig.savefig(image_out,dpi=180,bbox_inches='tight')
plt.show()
results=pd.DataFrame(rows)
display(results)
print(f'Coincidencias: {int(results.coincide.sum())}/4 (solo prueba ilustrativa, no exactitud de validación)')
csv_out='/content/TRL3_Prueba2_Predicciones_IA.csv'
results.to_csv(csv_out,index=False)
files.download(csv_out)
files.download(image_out)
